# LeRobotDataset v3(3.0) → v2.1 Export (OpenPI-safe `image`)

이 노트북은 LeRobotDataset **v3.0** 폴더(로컬 또는 Hub 스냅샷)를 읽어서 LeRobotDataset **v2.1 스타일(episode-per-file)** 폴더로 다시 저장합니다.

핵심 목표:
- v2.1 폴더 레이아웃(`meta/`, `data/`, optional `videos/`) 생성
- `data/**/episode_*.parquet` 안에 OpenPI가 기대하는 `image` 컬럼을 생성
- `image` 컬럼을 **dict(struct<bytes,path>)가 아니라** `uint8` 텐서형(고정크기 중첩 리스트)으로 저장해서 `torch.tensor(...)` 변환에서 크래시가 나지 않게 함

참고: 기존 방식처럼 `datasets.Image`(struct)로 `image`를 쓰면 `Could not infer dtype of dict` 에러가 날 수 있습니다.

In [1]:
# Dependencies check (run in an env that has these)
import sys
import importlib
import yaml
import os

def _req(name: str):
    try:
        return importlib.import_module(name)
    except Exception as e:
        raise RuntimeError(f'Missing dependency: {name} ({e})')

np = _req('numpy')
pa = _req('pyarrow')
pq = _req('pyarrow.parquet')
ds = _req('pyarrow.dataset')
PIL = _req('PIL')
Image = _req('PIL.Image')
tqdm = _req('tqdm.auto').tqdm

print('python:', sys.executable)
print('numpy:', np.__version__)
print('pyarrow:', pa.__version__)
print('Pillow:', PIL.__version__)

with open(r'../rby1-data-collection/config.yaml', encoding='utf-8') as f: # 추후 수정
    config = yaml.safe_load(f)

root = config['demo_root']
task_name = config['conversion_task_name']
user_name = config['user_name']
v3_dataset_saving_root = config['v3_dataset_saving_root']
v2_dataset_saving_root = config['v2_dataset_saving_root']

python: /home/hyunjin/miniforge3/envs/lerobot/bin/python
numpy: 2.2.6
pyarrow: 23.0.0
Pillow: 12.1.1


In [2]:
# ===== User config =====
from pathlib import Path
import json
import shutil

# (A) v3 input: either a local folder OR a hub snapshot folder.
# Local v3 example:
LOCAL_V3_DIR = Path(os.path.join(v3_dataset_saving_root, task_name)).resolve()
# Hub snapshot option (if you already downloaded via snapshot_download):
HUB_SNAPSHOT_DIR = None  # e.g. Path('../rby1-data-collection/Sample/_hf_check/<repo>/snapshots/<hash>').resolve()

# (B) v2.1 output folder
OUT_DIR = Path(os.path.join(v2_dataset_saving_root, task_name))

# If True, deletes OUT_DIR first (guarded by name check)
CLEAN_OUT_DIR = False

# Image tensor shape for OpenPI `image` column
IMAGE_HEIGHT = 224
IMAGE_WIDTH = 224

# Writes v2.1 `videos/` by slicing v3 videos (can be slow).
# OpenPI generally only needs the parquet `image` column, so you can keep this False.
WRITE_VIDEOS = False

print('LOCAL_V3_DIR:', LOCAL_V3_DIR)
print('HUB_SNAPSHOT_DIR:', HUB_SNAPSHOT_DIR)
print('OUT_DIR:', OUT_DIR)


LOCAL_V3_DIR: /media/hyunjin/T7/rby1_demo/LeRobotDataset_v3/PuttingCupintotheDishV2
HUB_SNAPSHOT_DIR: None
OUT_DIR: /media/hyunjin/T7/rby1_demo/LeRobotDataset_v2/PuttingCupintotheDishV2


In [3]:
# ===== Helpers =====
import io
import json
import shutil
from pathlib import Path

def _find_v3_root(dir_path: Path) -> Path:
    """Find v3 dataset root (must contain meta/info.json)."""
    dir_path = Path(dir_path)
    if (dir_path / 'meta/info.json').exists():
        return dir_path
    for p in dir_path.rglob('meta/info.json'):
        return p.parent.parent
    raise FileNotFoundError(f'Cannot find meta/info.json under: {dir_path}')

def _sorted_chunk_file_parquets(root: Path) -> list[Path]:
    root = Path(root)
    if not root.exists():
        return []
    return sorted([p for p in root.rglob('*.parquet') if p.is_file()])

def _guarded_rmtree(path: Path):
    path = Path(path)
    name = path.name.lower()
    if not any(tok in name for tok in ('export', 'v21', 'v2')):
        raise RuntimeError(
            'Refusing to delete OUT_DIR because its name does not contain export/v21/v2. '
            f'Got: {path}'
        )
    shutil.rmtree(path)

def _decode_image_struct(elem) -> 'Image.Image':
    """Decode HF datasets.Image-like element into a PIL Image."""
    if elem is None:
        raise ValueError('image element is None')
    if hasattr(elem, 'as_py'):
        elem = elem.as_py()
    if isinstance(elem, dict):
        raw = elem.get('bytes', None)
        path = elem.get('path', None)
        if raw is not None:
            return Image.open(io.BytesIO(raw)).convert('RGB')
        if path:
            return Image.open(path).convert('RGB')
    raise TypeError(f'Unsupported image element type: {type(elem)}')

def _images_to_fixed_chw_uint8(image_arr, height: int, width: int) -> pa.Array:
    """Convert struct<{bytes,path}> array to fixed-size nested lists uint8 [3,H,W]."""
    if isinstance(image_arr, pa.ChunkedArray):
        image_arr = image_arr.combine_chunks()
    if pa.types.is_fixed_size_list(image_arr.type) or pa.types.is_list(image_arr.type):
        # already list-like (assume numeric)
        return image_arr
    if not pa.types.is_struct(image_arr.type):
        raise TypeError(f'Expected struct/list image array, got: {image_arr.type}')

    py_elems = image_arr.to_pylist()
    chw_list = []
    for e in tqdm(py_elems, desc='decode+resize images'):
        img = _decode_image_struct(e)
        if (height is not None) and (width is not None):
            img = img.resize((width, height))
        arr = np.asarray(img, dtype=np.uint8)  # HWC
        if arr.ndim != 3 or arr.shape[2] != 3:
            raise ValueError(f'Expected HWC RGB image, got shape {arr.shape}')
        chw = np.transpose(arr, (2, 0, 1))  # CHW
        chw_list.append(chw)

    stacked = np.stack(chw_list, axis=0)  # N,C,H,W
    if stacked.shape[1] != 3 or stacked.shape[2] != height or stacked.shape[3] != width:
        raise ValueError(f'Unexpected stacked shape: {stacked.shape} (expected N,3,{height},{width})')

    flat = stacked.reshape(-1).tolist()
    values = pa.array(flat, type=pa.uint8())
    lvl_w = pa.FixedSizeListArray.from_arrays(values, width)
    lvl_h = pa.FixedSizeListArray.from_arrays(lvl_w, height)
    lvl_c = pa.FixedSizeListArray.from_arrays(lvl_h, 3)
    return lvl_c


def _zeros_fixed_chw_uint8(n: int, height: int, width: int) -> pa.Array:
    """Make fixed-size nested lists uint8 [3,H,W] filled with zeros, length n."""
    if n <= 0:
        # empty
        values = pa.array([], type=pa.uint8())
        lvl_w = pa.FixedSizeListArray.from_arrays(values, width)
        lvl_h = pa.FixedSizeListArray.from_arrays(lvl_w, height)
        lvl_c = pa.FixedSizeListArray.from_arrays(lvl_h, 3)
        return lvl_c

    stacked = np.zeros((n, 3, height, width), dtype=np.uint8)
    flat = stacked.reshape(-1).tolist()
    values = pa.array(flat, type=pa.uint8())
    lvl_w = pa.FixedSizeListArray.from_arrays(values, width)
    lvl_h = pa.FixedSizeListArray.from_arrays(lvl_w, height)
    lvl_c = pa.FixedSizeListArray.from_arrays(lvl_h, 3)
    return lvl_c


def _write_json(path: Path, obj):
    path = Path(path)
    path.write_text(json.dumps(obj, indent=2, ensure_ascii=False))

def _write_jsonl(path: Path, rows: list[dict]):
    path = Path(path)
    with path.open('w', encoding='utf-8') as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False))
            f.write('\n')

In [4]:
# ===== Resolve V3_DIR and load v3 metadata =====
V3_DIR = _find_v3_root(HUB_SNAPSHOT_DIR if HUB_SNAPSHOT_DIR else LOCAL_V3_DIR)
print('V3_DIR selected:', V3_DIR)

v3_info = json.loads((V3_DIR / 'meta/info.json').read_text())
robot_type = v3_info.get('robot_type', '')
fps = int(v3_info.get('fps', 10))
chunks_size_v21 = int(v3_info.get('chunks_size', 1000))

v3_features = v3_info.get('features', {})
image_keys = [k for k, v in v3_features.items() if isinstance(v, dict) and v.get('dtype') == 'image']
print('v3 image feature keys:', image_keys)

episodes_parquet_files = _sorted_chunk_file_parquets(V3_DIR / 'meta/episodes')
data_parquet_files = _sorted_chunk_file_parquets(V3_DIR / 'data')
if not episodes_parquet_files:
    raise FileNotFoundError(f'episodes parquet not found under: {V3_DIR / "meta/episodes"}')
if not data_parquet_files:
    raise FileNotFoundError(f'data parquet not found under: {V3_DIR / "data"}')

data_ds = ds.dataset([str(p) for p in data_parquet_files], format='parquet')
print('v3 data schema names (sample):', data_ds.schema.names[:30])

# Choose v3 image feature keys for OpenPI RBY1 config
# (dataset keys: image, left_wrist_image, right_wrist_image)

def _pick_key(keys: list[str], contains: str) -> str | None:
    contains = contains.lower()
    return next((k for k in keys if contains in str(k).lower()), None)

head_image_key = _pick_key(image_keys, 'head') or (image_keys[0] if image_keys else None)
left_image_key = _pick_key(image_keys, 'left')
right_image_key = _pick_key(image_keys, 'right')

if head_image_key is None:
    raise RuntimeError('No v3 image feature keys found; cannot build OpenPI-safe image columns')

print('head_image_key:', head_image_key)
print('left_image_key:', left_image_key)
print('right_image_key:', right_image_key)

V3_DIR selected: /media/hyunjin/T7/rby1_demo/LeRobotDataset_v3/PuttingCupintotheDishV2
v3 image feature keys: ['observation.images.head_rgb']
v3 data schema names (sample): ['observation.images.head_rgb', 'observation.state', 'action', 'is_first', 'is_last', 'is_terminal', 'timestamp', 'frame_index', 'episode_index', 'index', 'task_index']
head_image_key: observation.images.head_rgb
left_image_key: None
right_image_key: None


In [5]:
# ===== Read v3 episodes table (episode_index, start/length, tasks, etc.) =====
ep_tables = [pq.read_table(p) for p in episodes_parquet_files]
episodes_tbl = pa.concat_tables(ep_tables) if len(ep_tables) > 1 else ep_tables[0]
print('episodes columns:', episodes_tbl.column_names)

# Episode index column
ep_idx_col = None
for cand in ['episode_index', 'episode', 'episode_id', 'id']:
    if cand in episodes_tbl.column_names:
        ep_idx_col = cand
        break
if ep_idx_col is None:
    raise RuntimeError('Cannot find episode index column in episodes parquet')

episode_ids = sorted({int(x.as_py()) for x in episodes_tbl[ep_idx_col]})
print('episode_ids:', episode_ids)

# Optional tasks column (v3 sometimes has tasks as list/str)
tasks_col = None
for cand in ['tasks', 'task', 'task_name']:
    if cand in episodes_tbl.column_names:
        tasks_col = cand
        break
print('tasks_col:', tasks_col)

episodes columns: ['episode_index', 'tasks', 'length', 'data/chunk_index', 'data/file_index', 'dataset_from_index', 'dataset_to_index', 'stats/observation.images.head_rgb/min', 'stats/observation.images.head_rgb/max', 'stats/observation.images.head_rgb/mean', 'stats/observation.images.head_rgb/std', 'stats/observation.images.head_rgb/count', 'stats/observation.images.head_rgb/q01', 'stats/observation.images.head_rgb/q10', 'stats/observation.images.head_rgb/q50', 'stats/observation.images.head_rgb/q90', 'stats/observation.images.head_rgb/q99', 'stats/observation.state/min', 'stats/observation.state/max', 'stats/observation.state/mean', 'stats/observation.state/std', 'stats/observation.state/count', 'stats/observation.state/q01', 'stats/observation.state/q10', 'stats/observation.state/q50', 'stats/observation.state/q90', 'stats/observation.state/q99', 'stats/action/min', 'stats/action/max', 'stats/action/mean', 'stats/action/std', 'stats/action/count', 'stats/action/q01', 'stats/action/q

In [6]:
# ===== Export to v2.1 layout (episode-per-parquet) =====
if CLEAN_OUT_DIR and OUT_DIR.exists():
    _guarded_rmtree(OUT_DIR)

(OUT_DIR / 'meta').mkdir(parents=True, exist_ok=True)
(OUT_DIR / 'data').mkdir(parents=True, exist_ok=True)
if WRITE_VIDEOS:
    (OUT_DIR / 'videos').mkdir(parents=True, exist_ok=True)

# v2.1-ish features (drop v3 struct image cols; add OpenPI-safe `image`)
v21_features = {
    # ---- OpenPI (repack) expects these dataset keys ----
    'state': {'dtype': 'float32', 'shape': (16,), 'names': None},
    'actions': {'dtype': 'float32', 'shape': (16,), 'names': None},
    'prompt': {'dtype': 'string', 'shape': [1], 'names': None},

    'head_image': {
        'dtype': 'uint8',
        'shape': [3, IMAGE_HEIGHT, IMAGE_WIDTH],
        'names': ['channels', 'height', 'width'],
    },
    'left_wrist_image': {
        'dtype': 'uint8',
        'shape': [3, IMAGE_HEIGHT, IMAGE_WIDTH],
        'names': ['channels', 'height', 'width'],
    },
    'right_wrist_image': {
        'dtype': 'uint8',
        'shape': [3, IMAGE_HEIGHT, IMAGE_WIDTH],
        'names': ['channels', 'height', 'width'],
    },

    # ---- LeRobot default features (keep for compatibility) ----
    'timestamp': {'dtype': 'float32', 'shape': [1], 'names': None},
    'frame_index': {'dtype': 'int64', 'shape': [1], 'names': None},
    'episode_index': {'dtype': 'int64', 'shape': [1], 'names': None},
    'index': {'dtype': 'int64', 'shape': [1], 'names': None},
    'task_index': {'dtype': 'int64', 'shape': [1], 'names': None},

    # ---- Optional extras (kept for debugging / convenience) ----
    'observation.state': {'dtype': 'float32', 'shape': (16,), 'names': None},
    'action': {'dtype': 'float32', 'shape': (16,), 'names': None},
    'is_first': {'dtype': 'bool', 'shape': [1], 'names': None},
    'is_last': {'dtype': 'bool', 'shape': [1], 'names': None},
    'is_terminal': {'dtype': 'bool', 'shape': [1], 'names': None},
}

# NOTE: OpenPI가 쓰는 lerobot(v2.1)은 meta/info.json 스키마를 엄격하게 기대합니다.
# (lerobot.common.datasets.utils.create_empty_dataset_info 참고)
# 여기서는 video 없이(parquet에 image 텐서 포함) 가는 OpenPI-safe 모드로 설정합니다.

v21_info = {
    'codebase_version': 'v2.1',
    'robot_type': robot_type,
    'total_episodes': int(len(episode_ids)),
    'total_frames': 0,   # 아래에서 누적 후 업데이트
    'total_tasks': 0,    # tasks.jsonl 만든 후 업데이트
    'total_videos': 0,
    'total_chunks': 0,   # 아래에서 누적 후 업데이트
    'chunks_size': int(chunks_size_v21),
    'fps': int(fps),
    'splits': {'train': f'0:{len(episode_ids)}'},
    # v2.1 템플릿 (LeRobotDatasetMetadata.get_data_file_path에서 format()됨)
    'data_path': 'data/chunk-{episode_chunk:03d}/episode_{episode_index:06d}.parquet',
    'video_path': None,
    'features': v21_features,
}
_write_json(OUT_DIR / 'meta/info.json', v21_info)
print('Wrote:', OUT_DIR / 'meta/info.json')

# Copy tasks.parquet if present; also emit tasks.jsonl (normalize __index_level_* → task)
tasks_parquet_in = V3_DIR / 'meta/tasks.parquet'
task_index_to_name = {}
if tasks_parquet_in.exists():
    shutil.copy2(tasks_parquet_in, OUT_DIR / 'meta/tasks.parquet')
    try:
        tasks_tbl = pq.read_table(tasks_parquet_in)
        raw_rows = tasks_tbl.to_pylist()
        idx_keys = [k for k in tasks_tbl.column_names if str(k).startswith('__index_level_')]
        idx_key = idx_keys[0] if idx_keys else None

        tasks_jsonl_rows = []
        for i, row in enumerate(raw_rows):
            if not isinstance(row, dict):
                continue
            task_name = row.get('task') or row.get('task_name') or row.get('name')
            if (not task_name) and idx_key and (idx_key in row):
                task_name = row.get(idx_key)
            task_idx = row.get('task_index', i)
            try:
                task_idx = int(task_idx)
            except Exception:
                task_idx = i
            if task_name is None:
                task_name = f'task_{task_idx}'
            task_index_to_name[task_idx] = str(task_name)
            tasks_jsonl_rows.append({'task_index': task_idx, 'task': str(task_name)})

        if not tasks_jsonl_rows:
            tasks_jsonl_rows = [{'task_index': 0, 'task': 'unknown'}]
            task_index_to_name[0] = 'unknown'

        _write_jsonl(OUT_DIR / 'meta/tasks.jsonl', tasks_jsonl_rows)
        print('Wrote:', OUT_DIR / 'meta/tasks.jsonl')
    except Exception as e:
        print('Warning: failed to create normalized tasks.jsonl:', e)

# Ensure tasks.jsonl exists at least with one entry
if not (OUT_DIR / 'meta/tasks.jsonl').exists():
    _write_jsonl(OUT_DIR / 'meta/tasks.jsonl', [{'task_index': 0, 'task': 'unknown'}])
    task_index_to_name = {0: 'unknown'}
    print('Wrote:', OUT_DIR / 'meta/tasks.jsonl')

keep_cols = [
    'observation.state',
    'action',
    'is_first',
    'is_last',
    'is_terminal',
    'timestamp',
    'frame_index',
    'episode_index',
    'index',
    'task_index',
]

episodes_jsonl_rows = []
episodes_stats_jsonl_rows = []
dataset_cursor = 0
max_chunk_index = -1

# v3 episodes_tbl에서 stats를 가져와서 v2.1 episodes_stats.jsonl로 재구성
# output_key -> v3_stats_source_key 매핑
stats_source = {
    'observation.state': 'observation.state',
    'action': 'action',
    'actions': 'action',
    'is_first': 'is_first',
    'is_last': 'is_last',
    'is_terminal': 'is_terminal',
    'timestamp': 'timestamp',
    'frame_index': 'frame_index',
    'episode_index': 'episode_index',
    'index': 'index',
    'task_index': 'task_index',
    # OpenPI-safe head image 키
    'head_image': head_image_key,
}


def _get_v3_stat_row(episode_id: int):
    mask = pa.compute.equal(
        episodes_tbl[ep_idx_col],
        pa.scalar(episode_id, type=episodes_tbl[ep_idx_col].type),
    )
    match = episodes_tbl.filter(mask)
    if match.num_rows != 1:
        raise RuntimeError(f'Expected exactly 1 meta row for episode_index={episode_id}, got {match.num_rows}')
    return match


def _extract_stats(match_tbl: pa.Table) -> dict:
    out = {}
    for out_key, src_key in stats_source.items():
        # v3 stats column name: stats/<feature_key>/<stat_name>
        base = f'stats/{src_key}/'
        if f'{base}min' not in match_tbl.column_names:
            continue
        out[out_key] = {
            'min': match_tbl[f'{base}min'][0].as_py(),
            'max': match_tbl[f'{base}max'][0].as_py(),
            'mean': match_tbl[f'{base}mean'][0].as_py(),
            'std': match_tbl[f'{base}std'][0].as_py(),
            'count': match_tbl[f'{base}count'][0].as_py(),
        }
    return out


for episode_id in tqdm(episode_ids, desc='export episodes'):
    # Filter v3 frames for this episode
    tbl = data_ds.to_table(filter=(ds.field('episode_index') == episode_id))
    if 'index' in tbl.column_names:
        try:
            tbl = tbl.sort_by([('index', 'ascending')])
        except Exception:
            pass

    length = tbl.num_rows
    if length == 0:
        raise RuntimeError(f'No frames found for episode_index={episode_id}')

    # Build OpenPI-safe images for OpenPI RBY1 config
    image_col = _images_to_fixed_chw_uint8(tbl[head_image_key], IMAGE_HEIGHT, IMAGE_WIDTH)

    if left_image_key is not None and left_image_key in tbl.column_names:
        left_wrist_col = _images_to_fixed_chw_uint8(tbl[left_image_key], IMAGE_HEIGHT, IMAGE_WIDTH)
    else:
        left_wrist_col = _zeros_fixed_chw_uint8(length, IMAGE_HEIGHT, IMAGE_WIDTH)

    if right_image_key is not None and right_image_key in tbl.column_names:
        right_wrist_col = _images_to_fixed_chw_uint8(tbl[right_image_key], IMAGE_HEIGHT, IMAGE_WIDTH)
    else:
        right_wrist_col = _zeros_fixed_chw_uint8(length, IMAGE_HEIGHT, IMAGE_WIDTH)

    # Drop all v3 image columns to avoid struct/dict columns in output
    present_keep = [c for c in keep_cols if c in tbl.column_names]
    out_tbl = tbl.select(present_keep)

    # Add alias `actions` (copy of `action`)
    if 'action' in out_tbl.column_names and 'actions' not in out_tbl.column_names:
        out_tbl = out_tbl.append_column('actions', out_tbl['action'])

    # Add alias `state` (copy of `observation.state`)
    if 'observation.state' in out_tbl.column_names and 'state' not in out_tbl.column_names:
        out_tbl = out_tbl.append_column('state', out_tbl['observation.state'])

    # Add a default prompt column (must exist for OpenPI repack)
    if 'prompt' not in out_tbl.column_names:
        out_tbl = out_tbl.append_column('prompt', pa.array(['unknown'] * length, type=pa.string()))

    # Add OpenPI-safe image columns
    out_tbl = out_tbl.append_column('head_image', image_col)
    out_tbl = out_tbl.append_column('left_wrist_image', left_wrist_col)
    out_tbl = out_tbl.append_column('right_wrist_image', right_wrist_col)

    # IMPORTANT: v3 parquet에서 따라온 Arrow schema metadata(허깅페이스 features)가
    # v2.1 로더(datasets)가 이해 못하는 타입(List 등)을 포함할 수 있어서 제거합니다.
    out_tbl = out_tbl.replace_schema_metadata(None)

    chunk_index = episode_id // chunks_size_v21
    max_chunk_index = max(max_chunk_index, int(chunk_index))
    out_chunk_dir = OUT_DIR / 'data' / f'chunk-{chunk_index:03d}'
    out_chunk_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_chunk_dir / f'episode_{episode_id:06d}.parquet'
    pq.write_table(out_tbl, out_path, compression='zstd')

    # tasks (best-effort)
    tasks = None
    if tasks_col is not None:
        try:
            match = _get_v3_stat_row(int(episode_id))
            tval = match[tasks_col][0].as_py()
            if isinstance(tval, str):
                tasks = [tval]
            elif isinstance(tval, list):
                tasks = [str(x) for x in tval]
        except Exception:
            tasks = None
    if tasks is None and task_index_to_name and 'task_index' in out_tbl.column_names:
        try:
            tidx = int(out_tbl['task_index'][0].as_py())
            tasks = [task_index_to_name.get(tidx, f'task_{tidx}')]
        except Exception:
            tasks = None
    if tasks is None:
        tasks = ['unknown']

    episodes_jsonl_rows.append({
        'episode_index': int(episode_id),
        'length': int(length),
        'tasks': tasks,
    })

    # v2.1 episodes_stats.jsonl (OpenPI의 LeRobotDatasetMetadata가 요구)
    try:
        match = _get_v3_stat_row(int(episode_id))
        stats = _extract_stats(match)
    except Exception as e:
        print('[WARN] failed to extract v3 stats for episode', episode_id, e)
        stats = {}

    episodes_stats_jsonl_rows.append({
        'episode_index': int(episode_id),
        'stats': stats,
    })

    dataset_cursor += length

_write_jsonl(OUT_DIR / 'meta/episodes.jsonl', episodes_jsonl_rows)
print('Wrote:', OUT_DIR / 'meta/episodes.jsonl')

_write_jsonl(OUT_DIR / 'meta/episodes_stats.jsonl', episodes_stats_jsonl_rows)
print('Wrote:', OUT_DIR / 'meta/episodes_stats.jsonl')

# info.json totals 업데이트
v21_info['total_frames'] = int(dataset_cursor)
v21_info['total_chunks'] = int(max_chunk_index + 1) if max_chunk_index >= 0 else 0
v21_info['total_tasks'] = int(len(task_index_to_name) if task_index_to_name else v3_info.get('total_tasks', 0))
_write_json(OUT_DIR / 'meta/info.json', v21_info)
print('Updated totals in:', OUT_DIR / 'meta/info.json')

print('Export complete.')
print('OUT_DIR:', OUT_DIR)

Wrote: /media/hyunjin/T7/rby1_demo/LeRobotDataset_v2/PuttingCupintotheDishV2/meta/info.json
Wrote: /media/hyunjin/T7/rby1_demo/LeRobotDataset_v2/PuttingCupintotheDishV2/meta/tasks.jsonl


export episodes:   0%|          | 0/75 [00:00<?, ?it/s]

decode+resize images:   0%|          | 0/211 [00:00<?, ?it/s]

decode+resize images:   0%|          | 0/178 [00:00<?, ?it/s]

decode+resize images:   0%|          | 0/184 [00:00<?, ?it/s]

...

decode+resize images:   0%|          | 0/135 [00:00<?, ?it/s]

decode+resize images:   0%|          | 0/165 [00:00<?, ?it/s]

decode+resize images:   0%|          | 0/168 [00:00<?, ?it/s]

decode+resize images:   0%|          | 0/191 [00:00<?, ?it/s]

Wrote: /media/hyunjin/T7/rby1_demo/LeRobotDataset_v2/PuttingCupintotheDishV2/meta/episodes.jsonl
Wrote: /media/hyunjin/T7/rby1_demo/LeRobotDataset_v2/PuttingCupintotheDishV2/meta/episodes_stats.jsonl
Updated totals in: /media/hyunjin/T7/rby1_demo/LeRobotDataset_v2/PuttingCupintotheDishV2/meta/info.json
Export complete.
OUT_DIR: /media/hyunjin/T7/rby1_demo/LeRobotDataset_v2/PuttingCupintotheDishV2


In [7]:
# ===== Validation: make sure head/wrist images are NOT dict/struct =====
import pyarrow.parquet as pq

sample_parquets = _sorted_chunk_file_parquets(OUT_DIR / 'data')
assert sample_parquets, f'No parquet files written under: {OUT_DIR / "data"}'
sample_path = sample_parquets[0]
tbl = pq.read_table(sample_path)
print('sample parquet:', sample_path)
print('columns:', tbl.column_names)

for k in ['head_image', 'left_wrist_image', 'right_wrist_image']:
    assert k in tbl.column_names, f"Missing column: {k}"
    print(f'{k} type:', tbl[k].type)
    first = tbl[k][0].as_py()
    print(f'python type(first {k}):', type(first))
    if isinstance(first, dict):
        raise RuntimeError(f'Bad: {k} is still a dict')
    print(f'nested lengths (C,H,W) for {k}:', len(first), len(first[0]), len(first[0][0]))


sample parquet: /media/hyunjin/T7/rby1_demo/LeRobotDataset_v2/PuttingCupintotheDishV2/data/chunk-000/episode_000000.parquet
columns: ['observation.state', 'action', 'is_first', 'is_last', 'is_terminal', 'timestamp', 'frame_index', 'episode_index', 'index', 'task_index', 'actions', 'state', 'prompt', 'head_image', 'left_wrist_image', 'right_wrist_image']
head_image type: fixed_size_list<element: fixed_size_list<element: fixed_size_list<element: uint8>[224]>[224]>[3]
python type(first head_image): <class 'list'>
nested lengths (C,H,W) for head_image: 3 224 224
left_wrist_image type: fixed_size_list<element: fixed_size_list<element: fixed_size_list<element: uint8>[224]>[224]>[3]
python type(first left_wrist_image): <class 'list'>
nested lengths (C,H,W) for left_wrist_image: 3 224 224
right_wrist_image type: fixed_size_list<element: fixed_size_list<element: fixed_size_list<element: uint8>[224]>[224]>[3]
python type(first right_wrist_image): <class 'list'>
nested lengths (C,H,W) for right_w

In [8]:
# ===== Optional: upload OUT_DIR to Hugging Face Hub (dataset repo) =====
# Prereq: run `huggingface-cli login` in your terminal (you already did).

import importlib
from pathlib import Path

def _req2(name: str):
    try:
        return importlib.import_module(name)
    except Exception as e:
        raise RuntimeError(f'Missing dependency: {name} ({e}). Try: pip install {name.split(".")[0]}')

huggingface_hub = _req2('huggingface_hub')
HfApi = huggingface_hub.HfApi
create_repo = huggingface_hub.create_repo
upload_folder = huggingface_hub.upload_folder

# Change this
REPO_ID = f"{user_name}/{task_name}"+"V2"  
PRIVATE = False
COMMIT_MESSAGE = 'Add LeRobotDataset v2.1 export (OpenPI-safe image)'

out_dir = Path(OUT_DIR)
if not out_dir.exists():
    raise FileNotFoundError(f'OUT_DIR does not exist: {out_dir}')

# Create repo if needed (repo_type='dataset')
create_repo(repo_id=REPO_ID, repo_type='dataset', private=PRIVATE, exist_ok=True)
print('Repo ready:', REPO_ID)

# Upload. If you wrote videos and they are huge, consider ignore_patterns=['**/*.mp4']
res = upload_folder(
    repo_id=REPO_ID,
    repo_type='dataset',
    folder_path=str(out_dir),
    path_in_repo='',
    commit_message=COMMIT_MESSAGE,
    # ignore_patterns=['**/*.mp4'],
 )
print('Upload done:', res)

Repo ready: edipark/PuttingCupintotheDishV2V2


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Upload done: https://huggingface.co/datasets/edipark/PuttingCupintotheDishV2V2/tree/main/
